# 03 - Predator-Prey: an arms race

3 predators (red) chase 1 faster prey (green). Predators share a team reward for every touch; the prey is punished for it.

Training schedule: **for the first 0.5M steps the prey is frozen** (it doesn't learn - it barely moves). Then it starts learning too.
Watch what happens to the predators when their opponent stops being a fixed target. *(Pre-trained runs.)*

In [ ]:
%matplotlib inline
import matplotlib.pyplot as plt
from marl.analysis import (load_run, load_policy, evaluate, video, evolution_frames, plot_curves,
                           plot_matrix, moving_target_matrix, crossplay_matrix, speaker_messages,
                           message_separation, message_drift, pick)
from marl.render import show_video
plt.rcParams["figure.dpi"] = 90

In [ ]:
maddpg, iddpg = load_run("tag_maddpg"), load_run("tag_iddpg")
freeze = maddpg["config"]["freeze"]["prey"]
fig, ax = plt.subplots(1, 2, figsize=(11, 3.6))
plot_curves([maddpg, iddpg], "contacts", ax=ax[0], title="predator-prey touches per episode",
            vlines=[(freeze, "prey starts learning")])
plot_curves([maddpg, iddpg], "prey_min_dist", ax=ax[1], title="closest predator to the prey (avg)",
            vlines=[(freeze, "prey starts learning")])
plt.tight_layout()

## Evolution over training
Same episode, policies from later and later in training (both teams change).

In [ ]:
show_video(evolution_frames(maddpg, k=5), fps=10)

## Same predators, different prey
The final predators against (a) the frozen prey they first trained against and (b) the final, trained prey.

In [ ]:
from marl.analysis import MixedPolicy
final, early = load_policy(maddpg["ckpts"][-1]), load_policy(maddpg["ckpts"][2])
vs_early = MixedPolicy([final, final, final, early])     # agents: 3 predators, then the prey
vs_final = MixedPolicy([final, final, final, final])
for name, pol in (("vs frozen prey", vs_early), ("vs trained prey", vs_final)):
    print(f"{name:>16}: {evaluate(pol, 'tag', {}, 64)['contacts']:.2f} touches / episode")
show_video(video(vs_early, "tag", {}, title="final predators vs frozen prey") +
           video(vs_final, "tag", {}, title="final predators vs final prey"))

## Cross-play: is a later predator always a better predator?
Row *i*: predators from snapshot *i*. Column *j*: prey from snapshot *j*. Value: touches per episode (higher = better for predators).
If learning were a steady climb, every row would beat the rows above it against *every* prey. Look for rows that win against some prey and lose against others:
each side keeps adapting to the *current* opponent, not to opponents in general.

In [ ]:
M, steps = crossplay_matrix(maddpg, k=6, episodes=64)
plot_matrix(M, steps, title="MADDPG: touches per episode", xlabel="prey snapshot", ylabel="predator snapshot",
            cmap="viridis");